# Step 3. Find shifts. Currently they are adjusted from the ones Peter got.

In [ ]:
import numpy as np
import cupy as cp
import h5py
import matplotlib.pyplot as plt
import scipy
from holotomocupy.utils import *

## Init

In [ ]:
ntheta = 1800
ids = np.arange(0, 1800, 1800 / ntheta).astype('int')


path_out = '/data2/vnikitin/atomium_rec/20250607/AtomiumL1'
file_out = f'data.h5'


with h5py.File(f'{path_out}/data.h5') as fid:
    detector_pixelsize = fid['/exchange/detector_pixelsize'][0]    
    focustodetectordistance = fid['/exchange/focusdetectordistance'][0]    
    z1 = fid['/exchange/z1'][:] 
    theta = -fid['/exchange/theta'][ids, 0] / 180 * np.pi
    energy = fid['/exchange/energy'][0] 
    shifts = fid['/exchange/shifts'][ids]    
    shape = np.array(fid[f'/exchange/data0'].shape)

wavelength = 1.24e-09 / energy
z2 = focustodetectordistance - z1
magnifications = focustodetectordistance / z1
norm_magnifications = magnifications / magnifications[0]
distances = (z1 * z2) / focustodetectordistance * norm_magnifications**2
voxelsize = detector_pixelsize / magnifications[0]

n = shape[1]
ndist=len(z1)
print(f'{energy=}')
print(f'{z1=}')
print(f'{focustodetectordistance=}')
print(f'{detector_pixelsize=}')
print(f'{magnifications=}')
print(f'{voxelsize=}')
print(f'{distances=}')

### Motion shifts, alignment for a reference plane. Initially given with random shifts include, we subtract random shifts.

In [ ]:
import math
from holotomocupy.shift import Shift
with h5py.File(f'/data2/vnikitin/atomium/20250607/AtomiumL1/AtomiumL1_HT_014nm_/AtomiumL1_HT_014nm_rec_.nx','r') as fid:
    d = fid['/entry/data/data'][0:ntheta]        
norm_magnifications = np.array([1],dtype='float32')

n = d.shape[-1]
cl_shift = Shift(n, n,n,n, np.array([1],dtype='float32'),'float32')

correct3d_shifts = np.loadtxt(f'/data2/vnikitin/brain_rec/20251115/Y350a/correct_correct3D_1234.txt')[:4500][:, ::-1].astype('float32')
correct3d_shifts = np.tile(correct3d_shifts[:, np.newaxis], (1, ndist, 1))

ids = np.arange(0, 4500, 4500 / ntheta).astype('int')
correct3d_shifts = correct3d_shifts[ids]
correct3d_shifts*=20/14
# correct3d_shifts[...,1] = 0
pos = -correct3d_shifts[:,0]
pos[...,1] *= 0
pos[...,1] += -(-14.621)
plt.plot(pos[...,1],'.')
plt.plot(pos[...,0],'.')
plt.show()
nchunk = 32
sd = np.empty_like(d)
for k in range(0,math.ceil(len(d)/nchunk)):    
    st = k*nchunk
    end = min((k+1)*nchunk,len(d))
    dd = cp.array(d[st:end])
    ppos = cp.array(pos[st:end])
    
    sd[st:end] = cl_shift.curlyS(dd,ppos,0).get()    

from types import SimpleNamespace
from holotomocupy.tomo_batched import TomoBatched
args = SimpleNamespace()
args.ngpus = cp.cuda.runtime.getDeviceCount()

args.ntheta = ntheta
args.nobj = n
args.nzobj = n

args.nchunk = 1
args.show = True
args.theta = theta
args.mask_r = 1.2

cl_rec = TomoBatched(args)

import dxchange

for it in [32,64,128,256]:
    rec = cl_rec.rec_tomo(sd.astype('complex64'), it).astype('float32')
    dxchange.write_tiff_stack(rec,f'/data2/vnikitin/atomium_rec/20250607/AtomiumL1/rec_peter{it}/r',overwrite=True)
